In [2]:
# import libraries
import os
import sys
sys.path.insert(0, '..')

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif

from src.data.load_data import load_raw_data, encode_target
from src.data.clean_data import clean_pipeline
from src.features.build_features import build_features_pipeline

In [3]:
# load raw data
data = encode_target(load_raw_data('../data/raw/bank-full.csv'))
cleaned = clean_pipeline(data)
print("Data loaded and cleaned successfully.")
print(f"Data shape: {cleaned.shape}")

Data loaded and cleaned successfully.
Data shape: (45189, 17)


In [4]:
train_data, temp_data = train_test_split(cleaned, test_size=0.2, random_state=42, stratify=cleaned['y'])  # train-test split
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42, stratify=temp_data['y'])  # validation-test split
print(f"Training data shape: {train_data.shape}")
print(f"Validation data shape: {val_data.shape}")
print(f"Test data shape: {test_data.shape}")

train_features, train_target = build_features_pipeline(train_data, {'val' : val_data, 'test' : test_data})
new_columns = [c for c in train_features.columns if c not in train_data.columns]
print(f"New columns created: {new_columns}")
train_features[new_columns].head()

Training data shape: (36151, 17)
Validation data shape: (4519, 17)
Test data shape: (4519, 17)
New columns created: ['contact_recency_score', 'campaign_intensity', 'age_life_stage', 'is_high_season', 'contact_channel_trust', 'seasonal_conversion_prior']


,contact_recency_score,campaign_intensity,age_life_stage,is_high_season,contact_channel_trust,seasonal_conversion_prior
43859,0.010753,0.500000,pre_retirement,0,cellular_success,0.104108
9581,0.000000,1.000000,pre_retirement,0,unknown_unknown,0.104108
30225,0.000000,1.000000,retired,0,cellular_unknown,0.168750
36088,0.001425,0.166667,family_formation,0,cellular_other,0.066643
26392,0.000000,1.000000,pre_retirement,0,cellular_unknown,0.100955


In [6]:
# Confirm the seasonal prior lookup came from TRAIN only
month_rate_in_train = train_features.groupby('month')['seasonal_conversion_prior'].first()
actual_train_target_rate_by_month = train_features.groupby('month')['y'].mean()
comparison_df = pd.DataFrame({'Feature_value': month_rate_in_train, 'Actual_Train_Rate': actual_train_target_rate_by_month})
print(comparison_df)

       Feature_value  Actual_Train_Rate
month                                  
apr         0.193992           0.193992
aug         0.109952           0.109952
dec         0.464088           0.464088
feb         0.168750           0.168750
jan         0.102770           0.102770
jul         0.092438           0.092438
jun         0.104108           0.104108
mar         0.511749           0.511749
may         0.066643           0.066643
nov         0.100955           0.100955
oct         0.437288           0.437288
sep         0.458150           0.458150


In [7]:
# Mutual information: which features carry the most signal about the target
numeric_check_cols = ['age','balance','campaign', 'previous', 'days_since_contact',
                      'contact_recency_score', 'campaign_intensity',
                      'seasonal_conversion_prior', 'is_high_season']
mi_scores = mutual_info_classif(train_features[numeric_check_cols], train_features['y'], random_state=42)
pd.Series(mi_scores, index=numeric_check_cols).sort_values(ascending=False)

contact_recency_score        0.033331
seasonal_conversion_prior    0.027023
days_since_contact           0.026874
balance                      0.023485
campaign_intensity           0.020169
is_high_season               0.018095
previous                     0.009968
age                          0.008813
campaign                     0.006136
dtype: float64

In [8]:
# Cramer's V: association strength between categorical features and the target
from scipy.stats import chi2_contingency
def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    min_dim = min(confusion_matrix.shape) - 1
    return (chi2 / n*min_dim) ** 0.5

for col in ['poutcome', 'job', 'contact', 'age_life_stage', 'contact_channel_trust']:
    v = cramers_v(train_features[col], train_features['y'])
    print(f"Cramer's V between {col} and target: {v:.4f}")

Cramer's V between poutcome and target: 0.3138
Cramer's V between job and target: 0.1374
Cramer's V between contact and target: 0.1492
Cramer's V between age_life_stage and target: 0.1625
Cramer's V between contact_channel_trust and target: 0.3320
